In [0]:
import os, sys
import pandas as pd
from pyspark.sql import functions as F

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, repo_root)

from src import config_loader as cl
from src import silver as sv

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("resource", "ALL", "Resource (ALL or e.g. Patient)")
dbutils.widgets.text("load_dates", "", "Bronze load dates, comma separated (blank = all)")
dbutils.widgets.text("catalog", "", "Catalog override (optional)")

g = lambda k: dbutils.widgets.get(k).strip()
load_dates = [d.strip() for d in g("load_dates").split(",") if d.strip()] or None
overrides = {"project": {"catalog": g("catalog")}} if g("catalog") else None
cfg = cl.load_config(os.path.join(repo_root, "config", "config.yaml"), overrides)

only = None if g("resource").upper() == "ALL" else [g("resource")]
resources = cl.get_resources(cfg, only)
print("Resources:", [r["name"] for r in resources], "| load_dates:", load_dates or "ALL")

In [0]:
silver_dfs, rows = {}, []
for r in resources:
    n = r["name"]
    bronze_rows = sv._bronze(spark, cfg, n, load_dates).count()
    parse_fail = sv.count_parse_failures(spark, cfg, n, load_dates)
    df = sv.build_silver_df(spark, cfg, n, load_dates)
    silver_dfs[n] = df
    silver_rows = df.count()
    rows.append({"resource": n, "bronze_rows": bronze_rows, "silver_rows": silver_rows,
                 "removed_as_duplicate_or_no_id": bronze_rows - silver_rows,
                 "json_parse_failures": parse_fail})
display(pd.DataFrame(rows))

In [0]:
for n, df in silver_dfs.items():
    print(f"=== {n} ===")
    display(df.drop("source_batch_id", "api_url_or_params", "row_hash").limit(10))

In [0]:
for n, df in silver_dfs.items():
    print(f"=== {n}: share of NULLs per column ===")
    display(pd.DataFrame(sv.null_profile(df)))

In [0]:
checks = []
for r in resources:
    n = r["name"]; key = r["silver_key"]
    df = silver_dfs[n]
    dups = (df.groupBy(key, F.coalesce("meta_version_id", F.lit("unknown"))).count()
              .where("count > 1").count())
    checks.append({"check": f"{n}: duplicate (id, version) rows", "result": dups, "expected": 0})
display(pd.DataFrame(checks))

# Orphans: child rows pointing at a parent that is not in this Silver data.
# Only meaningful when all four resources were built (resource = ALL).
if len(silver_dfs) == 4:
    def orphans(child, fk, parent, pk):
        c = silver_dfs[child].where(F.col(fk).isNotNull())
        p = silver_dfs[parent].select(F.col(pk).alias("_pk")).distinct()
        total = c.count()
        n = c.join(p, c[fk] == p["_pk"], "left_anti").count()
        return {"relationship": f"{child}.{fk} -> {parent}.{pk}",
                "rows_with_reference": total, "parent_not_in_silver": n,
                "pct_orphan": round(100.0 * n / total, 1) if total else None}
    display(pd.DataFrame([
        orphans("Encounter", "patient_id", "Patient", "patient_id"),
        orphans("Observation", "patient_id", "Patient", "patient_id"),
        orphans("Observation", "encounter_id", "Encounter", "encounter_id"),
        orphans("Condition", "patient_id", "Patient", "patient_id"),
        orphans("Condition", "encounter_id", "Encounter", "encounter_id"),
    ]))